In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch11-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "11-03-shap.py"

# 從背景分子到環己酮的預測

ESOL 第一次執行會自動下載。需要自行修改或添加分析時，將程式接在寫入分析腳本的儲存格末端，重新執行該格與最末一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# ESOL，1,128 筆。水溶解度 log10(mol/L)，DOI 10.1021/ci034243x。
_, (train, valid, test), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
names = ["MolLogP", "MolWt", "TPSA", "NumRotatableBonds"]
functions = [getattr(Descriptors, name) for name in names]

def describe(dataset):
    mols = [Chem.MolFromSmiles(s) for s in dataset.ids]
    return np.array([[fn(mol) for fn in functions] for mol in mols])

X_train, X_valid = describe(train), describe(valid)
y_train, y_valid = train.y.ravel(), valid.y.ravel()
model = RandomForestRegressor(n_estimators=120, min_samples_leaf=3, random_state=42)
model.fit(X_train, y_train)
prediction = model.predict(X_valid)
print(f"train/valid/test: {len(train)}/{len(valid)}/{len(test)}")
print(f"validation MAE = {mean_absolute_error(y_valid, prediction):.3f}")
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)

from itertools import combinations, permutations

rng = np.random.default_rng(42)
background = X_train[rng.choice(len(train), size=32, replace=False)]

def explain(rows, background):
    # 對每個欄位組合保留待解釋分子的值，其餘欄位來自同一背景列。
    d = rows.shape[1]
    values = {}
    for size in range(d + 1):
        for subset in combinations(range(d), size):
            mixed = np.tile(background, (len(rows), 1, 1))
            for j in subset:
                mixed[:, :, j] = rows[:, j, None]
            values[frozenset(subset)] = model.predict(
                mixed.reshape(-1, d)).reshape(len(rows), len(background)).mean(axis=1)
    # 四欄有 24 種揭露順序，將每欄帶來的預測差取平均。
    phi = np.zeros_like(rows)
    orders = list(permutations(range(d)))
    for order in orders:
        known = frozenset()
        for j in order:
            updated = known | {j}
            phi[:, j] += values[updated] - values[known]
            known = updated
    phi /= len(orders)
    base = values[frozenset()][0]
    np.testing.assert_allclose(base + phi.sum(axis=1), model.predict(rows), atol=1e-10)
    return base, phi

base, phi = explain(X_valid, background)
smiles = [s.strip() for s in valid.ids]
row = smiles.index("O=C1CCCCC1")
print(f"baseline = {base:.4f}, prediction = {prediction[row]:.4f}, measured = {y_valid[row]:.4f}")
for name, feature, local, global_mean in zip(names, X_valid[row], phi[row], abs(phi).mean(axis=0)):
    print(f"{name:18s} value {feature:.4f}  SHAP {local:+.4f}  mean |SHAP| {global_mean:.4f}")

# MolLogP 和 MolWt 先後加入時，每一欄的預測差。
for known in [[], [0], [1], [0, 1]]:
    mixed = background.copy()
    mixed[:, known] = X_valid[row, known]
    print([names[j] for j in known], f"{model.predict(mixed).mean():.4f}")

fig, (local_ax, global_ax) = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
positions = np.r_[base, base + np.cumsum(phi[row])]
for i, (a, b) in enumerate(zip(positions, positions[1:])):
    local_ax.barh(i, abs(b - a), left=min(a, b), height=.55,
                  color="#33768a" if b > a else "#bf7245")
    if i < len(names) - 1:
        local_ax.vlines(b, i + .275, i + .725, colors=".55", linestyles="dotted")
local_ax.scatter([base, prediction[row]], [0, 3], color="black", zorder=3, s=16)
local_ax.set(yticks=range(4), yticklabels=names, xlabel="Predicted log S",
             title=f"A   Cyclohexanone: {base:.2f} to {prediction[row]:.2f}")
global_ax.barh(range(4), abs(phi).mean(axis=0), color="#62788b", height=.55)
global_ax.set(yticks=range(4), yticklabels=names, xlabel="Mean |SHAP| (log S units)",
              title="B   All validation rows")
for ax in (local_ax, global_ax):
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
fig.savefig(out / "11-03-shap.png", dpi=165)


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "11-03-shap.png")))